# Clase 070 — Gradient Descent: batch, stochastic, mini-batch

Implementamos las tres variantes de gradient descent **a mano** con NumPy sobre un dataset lineal, comparamos sus curvas de costo, vemos el efecto del *learning rate* y por qué el *feature scaling* es obligatorio antes de `SGDRegressor`.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset sintético

$y = 4 + 3x + \text{ruido}$, con $m=200$. El $\theta$ verdadero es $[4, 3]$; lo usaremos para verificar que las tres variantes convergen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
m = 200
X = 2 * np.random.rand(m, 1)
y = 4 + 3 * X[:, 0] + np.random.randn(m)
X_b = np.c_[np.ones((m, 1)), X]
theta_true = np.array([4., 3.])
print('m =', m, '| X_b', X_b.shape)

## 2. Batch GD

Cada paso usa **todas** las muestras: $\nabla = \frac{2}{m}X^T(X\theta - y)$. Convergencia suave pero $O(m)$ por iteración.

In [ ]:
def batch_gd(X_b, y, eta=0.1, n_iter=1000, seed=0):
    rng = np.random.default_rng(seed)
    theta = rng.standard_normal(X_b.shape[1])
    m = len(y); cost = []
    for _ in range(n_iter):
        grad = (2 / m) * X_b.T @ (X_b @ theta - y)
        theta -= eta * grad
        cost.append(np.mean((X_b @ theta - y) ** 2))
    return theta, cost

theta_bgd, cost_bgd = batch_gd(X_b, y)
print('BGD theta:', theta_bgd.round(3))
assert np.allclose(theta_bgd, theta_true, atol=0.3)
print('OK: batch GD converge a [4, 3]')

## 3. Stochastic GD

Cada paso usa **una sola** muestra al azar. Es ruidoso, así que baja el learning rate con el tiempo mediante un *learning schedule* $\eta_t = t_0/(t_1+t)$.

In [ ]:
def stochastic_gd(X_b, y, n_epochs=50, t0=5.0, t1=50.0, seed=0):
    rng = np.random.default_rng(seed)
    theta = rng.standard_normal(X_b.shape[1])
    m = len(y); cost = []
    for epoch in range(n_epochs):
        for i in range(m):
            idx = rng.integers(m)
            xi, yi = X_b[idx:idx+1], y[idx:idx+1]
            grad = 2 * xi.T @ (xi @ theta - yi)
            eta = t0 / (t1 + epoch * m + i)          # learning schedule
            theta -= eta * grad.ravel()
        cost.append(np.mean((X_b @ theta - y) ** 2))
    return theta, cost

theta_sgd, cost_sgd = stochastic_gd(X_b, y)
print('SGD theta:', theta_sgd.round(3))
assert np.allclose(theta_sgd, theta_true, atol=0.6)
print('OK: stochastic GD converge (con más ruido) a [4, 3]')

## 4. Mini-batch GD y comparación de curvas

Usa lotes de tamaño intermedio: menos ruido que SGD, más liviano que BGD. Comparamos las tres curvas de costo (escala log).

In [ ]:
def minibatch_gd(X_b, y, n_epochs=50, batch_size=20, eta=0.05, seed=0):
    rng = np.random.default_rng(seed)
    theta = rng.standard_normal(X_b.shape[1])
    m = len(y); cost = []
    for _ in range(n_epochs):
        idx = rng.permutation(m)
        for start in range(0, m, batch_size):
            b = idx[start:start+batch_size]
            grad = (2 / len(b)) * X_b[b].T @ (X_b[b] @ theta - y[b])
            theta -= eta * grad
        cost.append(np.mean((X_b @ theta - y) ** 2))
    return theta, cost

theta_mb, cost_mb = minibatch_gd(X_b, y)
for name, th in [('BGD', theta_bgd), ('SGD', theta_sgd), ('Mini-batch', theta_mb)]:
    assert np.allclose(th, theta_true, atol=0.6), name
    print(f'{name:11s} -> {th.round(3)}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cost_bgd[:50], label='BGD')
ax.plot(cost_sgd, label='SGD')
ax.plot(cost_mb, label='Mini-batch')
ax.set_xlabel('epoch'); ax.set_ylabel('MSE'); ax.set_yscale('log'); ax.legend()
ax.set_title('Curvas de costo: BGD vs SGD vs Mini-batch')
plt.tight_layout(); plt.show()

## 5. Efecto del learning rate

Con $\eta$ muy chico la convergencia es lentísima; con $\eta$ muy grande el costo oscila o diverge.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for eta in [0.001, 0.01, 0.1, 0.5]:
    _, cost = batch_gd(X_b, y, eta=eta, n_iter=100)
    ax.plot(cost, label=f'η={eta}')
ax.set_xlabel('iteración'); ax.set_ylabel('MSE'); ax.set_ylim(0, 50); ax.legend()
ax.set_title('Efecto del learning rate en la convergencia')
plt.tight_layout(); plt.show()
print('η=0.001 converge lentísimo; η alto puede oscilar o diverger')

## 6. Feature scaling con `SGDRegressor`

Sobre `load_diabetes` exageramos la escala de una feature (×1000). Sin `StandardScaler`, SGD no converge; con él, sí. (California Housing requiere descarga, así que usamos diabetes, incluido en sklearn.)

In [ ]:
import warnings
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline

Xd, yd = load_diabetes(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.2, random_state=42)
escala = np.ones(Xd.shape[1]); escala[2] = 1000.0        # una feature domina el gradiente
Xtr_raw, Xte_raw = Xtr * escala, Xte * escala

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    sgd_no = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42).fit(Xtr_raw, ytr)
    pipe = make_pipeline(StandardScaler(),
                         SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)).fit(Xtr_raw, ytr)

print('sin scaler -> R2 test:', round(sgd_no.score(Xte_raw, yte), 4), '| n_iter_:', sgd_no.n_iter_)
print('con scaler -> R2 test:', round(pipe.score(Xte_raw, yte), 4),
      '| n_iter_:', pipe.named_steps['sgdregressor'].n_iter_)
assert pipe.score(Xte_raw, yte) > sgd_no.score(Xte_raw, yte)
print('OK: con StandardScaler el SGD generaliza mucho mejor')

## 7. `SGDRegressor` vs `LinearRegression`

Sobre diabetes escalado, SGD (aproximación estocástica) debe dar coeficientes muy parecidos a la solución cerrada.

In [ ]:
from sklearn.linear_model import LinearRegression

scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
lin = LinearRegression().fit(Xtr_s, ytr)
sgd = SGDRegressor(max_iter=5000, tol=1e-4, random_state=42).fit(Xtr_s, ytr)
print('LinearRegression R2:', round(lin.score(Xte_s, yte), 4))
print('SGDRegressor     R2:', round(sgd.score(Xte_s, yte), 4))
# diabetes tiene features correlacionadas: comparamos las PREDICCIONES, no los coeficientes uno a uno
corr_pred = np.corrcoef(lin.predict(Xte_s), sgd.predict(Xte_s))[0, 1]
print('correlación de predicciones:', round(corr_pred, 4))
assert corr_pred > 0.98 and abs(lin.score(Xte_s, yte) - sgd.score(Xte_s, yte)) < 0.05
print('OK: SGD aproxima la solución cerrada (mismas predicciones y R2)')

## Ejercicios

1. **SGD sin schedule.** Quitá el learning schedule (usá $\eta$ constante) en `stochastic_gd` y observá cómo el costo nunca se asienta: oscila para siempre alrededor del mínimo.
2. **Divergencia.** Corré `batch_gd` con $\eta=1.0$ y verificá que el costo crece en lugar de bajar. ¿A partir de qué $\eta$ empieza a diverger en este dataset?
3. **Tamaño de mini-batch.** Barré `batch_size ∈ {1, 8, 32, 200}` y compará las curvas de costo. ¿Cuál es más ruidosa y cuál más suave?
4. **`partial_fit` online.** Entrená un `SGDRegressor` llamando `partial_fit` en un loop y graficá la loss por epoch, instanciando el modelo **fuera** del loop.

## Conclusiones

- Las tres variantes minimizan el mismo MSE; se diferencian en el **costo por paso** y en el **ruido** de la trayectoria.
- El **learning schedule** es lo que permite a SGD asentarse cerca del mínimo pese al ruido.
- Sin **feature scaling**, SGD zigzaguea por un valle alargado y puede no converger: `StandardScaler` es prerrequisito.
- `SGDRegressor` aproxima muy bien la solución cerrada de `LinearRegression` cuando los datos están escalados.